In [2]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

!pip install koreanize-matplotlib -q
import koreanize_matplotlib

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import joblib, json

# ── 경로 설정 ──
BASE        = '..'
DATA_PATH   = f'{BASE}/data/raw/train.csv'
MODELS_PATH = f'{BASE}/saved_models'
IMG_PATH    = f'{BASE}/images'

train = pd.read_csv(DATA_PATH)
y = train['임신 성공 여부'].values


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ── 1. 트리 모델 Feature Importance ──

def get_lgbm_importance(model_path_pattern, n_folds=10):
    importances = []
    feat_names = None
    for fold in range(1, n_folds + 1):
        model = joblib.load(f'{model_path_pattern}_fold{fold}.pkl')
        if feat_names is None:
            feat_names = model.booster_.feature_name()
        importances.append(model.booster_.feature_importance(importance_type='gain'))
    return np.mean(importances, axis=0), feat_names

def get_xgb_importance(model_path_pattern, feat_names, n_folds=10):
    importances = []
    for fold in range(1, n_folds + 1):
        model = joblib.load(f'{model_path_pattern}_fold{fold}.pkl')
        scores = model.get_booster().get_score(importance_type='gain')
        # feat_names 기준으로 맞춰서 없는 피처는 0으로 채움
        importances.append([scores.get(f, 0) for f in feat_names])
    return np.mean(importances, axis=0)

# lgbm 먼저 로드해서 feat_names 기준 확보
lgbm_imp, feat_names = get_lgbm_importance(f'{MODELS_PATH}/exp_tuned_lgbm_seed42')

# xgb, cat은 feat_names 넘겨서 길이 맞춤
xgb_imp = get_xgb_importance(f'{MODELS_PATH}/exp_tuned_xgb_seed42', feat_names)
cat_imp, _ = get_cat_importance(f'{MODELS_PATH}/exp_tuned_cat_seed42')

print(f'LGBM: {lgbm_imp.shape}, XGB: {xgb_imp.shape}, CAT: {cat_imp.shape}')

def get_cat_importance(model_path_pattern, n_folds=10):
    importances = []
    feat_names = None
    for fold in range(1, n_folds + 1):
        model = joblib.load(f'{model_path_pattern}_fold{fold}.pkl')
        if feat_names is None:
            feat_names = model.feature_names_
        importances.append(model.get_feature_importance())
    return np.mean(importances, axis=0), feat_names

lgbm_imp, feat_names = get_lgbm_importance(f'{MODELS_PATH}/exp_tuned_lgbm_seed42')
xgb_imp,  _          = get_xgb_importance(f'{MODELS_PATH}/exp_tuned_xgb_seed42')
cat_imp,  _          = get_cat_importance(f'{MODELS_PATH}/exp_tuned_cat_seed42')

print(f'피처 수: {len(feat_names)}')
print(f'LGBM imp shape: {lgbm_imp.shape}')
print(f'XGB  imp shape: {xgb_imp.shape}')
print(f'CAT  imp shape: {cat_imp.shape}')

피처 수: 75
LGBM imp shape: (75,)
XGB  imp shape: (68,)
CAT  imp shape: (75,)


In [ ]:
# ── 2. 트리 모델 Top 20 Feature Importance 시각화 ──

def plot_importance(importance, feature_names, title, ax, top_n=20):
    df = pd.DataFrame({'feature': feature_names, 'importance': importance})
    df = df.sort_values('importance', ascending=False).head(top_n)
    sns.barplot(data=df, x='importance', y='feature', ax=ax, palette='viridis')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance (Gain)')
    ax.set_ylabel('')

fig, axes = plt.subplots(1, 3, figsize=(20, 10))
plot_importance(lgbm_imp, feat_names, 'LightGBM Feature Importance', axes[0])
plot_importance(xgb_imp,  feat_names, 'XGBoost Feature Importance',  axes[1])
plot_importance(cat_imp,  feat_names, 'CatBoost Feature Importance', axes[2])

plt.tight_layout()
plt.savefig(f'{IMG_PATH}/feature_importance_tree.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료: feature_importance_tree.png')

LGBM: (75,), XGB: (75,), CAT: (75,)


In [ ]:
# ── 3. FT-Transformer Attention Weight 시각화 ──
# FTT는 attention weight를 직접 추출하기 어려우므로
# OOF 예측값과 각 피처의 상관관계로 대체 (proxy importance)

ftt_oof = np.load(f'{MODELS_PATH}/exp25_fttransformer_seed42_oof.npy')

# 전처리된 피처 데이터 로드
from src.preprocess import preprocess
from src.features import add_features

X = preprocess(train.drop(columns=['임신 성공 여부']))
X = add_features(X)

ftt_proxy = {}
for col in X.columns:
    try:
        corr = np.corrcoef(X[col].values, ftt_oof)[0, 1]
        ftt_proxy[col] = abs(corr)
    except:
        ftt_proxy[col] = 0

ftt_imp = np.array([ftt_proxy[col] for col in X.columns])
ftt_feat_names = list(X.columns)

fig, ax = plt.subplots(figsize=(8, 10))
plot_importance(ftt_imp, ftt_feat_names, 'FT-Transformer Proxy Importance\n(|corr| with OOF)', ax)
plt.tight_layout()
plt.savefig(f'{IMG_PATH}/feature_importance_ftt.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료: feature_importance_ftt.png')

In [ ]:
# ── 4. SAINT Proxy Importance ──

saint_oof = np.load(f'{MODELS_PATH}/exp26_saint_seed42_oof.npy')

saint_proxy = {}
for col in X.columns:
    try:
        corr = np.corrcoef(X[col].values, saint_oof)[0, 1]
        saint_proxy[col] = abs(corr)
    except:
        saint_proxy[col] = 0

saint_imp = np.array([saint_proxy[col] for col in X.columns])

fig, ax = plt.subplots(figsize=(8, 10))
plot_importance(saint_imp, ftt_feat_names, 'SAINT Proxy Importance\n(|corr| with OOF)', ax)
plt.tight_layout()
plt.savefig(f'{IMG_PATH}/feature_importance_saint.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료: feature_importance_saint.png')